In [121]:
import os
import nltk
import re
from collections import Counter
from nltk.metrics.distance import edit_distance

HOME_DIR = "."
CORPUS_PT_DIR = os.path.join(HOME_DIR, "corpus_pt")

os.makedirs(CORPUS_PT_DIR, exist_ok=True)

### Funcao apenas para ler o arquivo de texto
1. Le o arquivo `texto.txt` e retorna uma lista de linhas do arquivo

In [122]:
def read_text_file(path):
    try:
        with open(path, encoding="utf-8-sig") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(path, encoding="latin-1") as f:
            return f.read()
        
capivara_path = os.path.join(CORPUS_PT_DIR, "capivara-pt.txt")
capivara_text = read_text_file(capivara_path)

### **Tokenização**
- Uso uma regex que separa as palavras e a pontuacao certinho

In [ ]:
#_LETTERS = "A-Za-zÀ-ÖØ-öø-ÿ"
_REGEX = r"[A-Za-zÀ-ÖØ-öø-ÿ]+(?:['-][A-Za-zÀ-ÖØ-öø-ÿ]+)*"
_REGEX_EXEMPLE = r"[a-zA-ZÀ-ÿ]+(?:[''-][a-zA-ZÀ-ÿ]+)*|[^\w\s]|\d+" #exemplo outra regex
_TOKEN_REGEX = re.compile(rf"{_REGEX}|\S")

def tokenize(text):
    return _TOKEN_REGEX.findall(text)

# texto
texto_ = capivara_text
tokens_ = tokenize(texto_)
print(f"Texto de exemplo: {len(texto_)} caracteres")
print(f"Total de tokens: {len(tokens_)}")
print("Primeiros 50 tokens:", tokens_[:50])

Texto de exemplo: 1173 caracteres
Total de tokens: 235
Primeiros 50 tokens: ['A', 'capivara', '(', 'nome', 'científico', ':', 'hydrochoerus', 'hydrochaeris', ')', 'é', 'uma', 'espécie', 'de', 'mamífero', 'roedor', 'da', 'família', 'caviidae', 'e', 'subfamília', 'hydrochoerinae', '.', 'alguns', 'autores', 'consideram', 'que', 'deva', 'ser', 'classificada', 'em', 'uma', 'família', 'própria', '.', 'está', 'incluída', 'no', 'mesmo', 'grupo', 'de', 'roedores', 'ao', 'qual', 'se', 'classificam', 'as', 'pacas', ',', 'cutias', ',']


### 2.1 Converte todos os tokens para letras minusculas

In [124]:
def to_lowercase(tokens):
    token_lower = list()
    for t in tokens:
        tkl = t.lower()
        token_lower.append(tkl)
        
        
    return token_lower


tokens_lower = to_lowercase(tokens_)
print(tokens_lower[:30])

['a', 'capivara', '(', 'nome', 'científico', ':', 'hydrochoerus', 'hydrochaeris', ')', 'é', 'uma', 'espécie', 'de', 'mamífero', 'roedor', 'da', 'família', 'caviidae', 'e', 'subfamília', 'hydrochoerinae', '.', 'alguns', 'autores', 'consideram', 'que', 'deva', 'ser', 'classificada', 'em']


### 2.2 Remove a pontuacao

In [ ]:
_HAS_ALNUM = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]") #regex de alfanumerico -> ao menos uma letra ou numero

def remove_punctuation(tokens):
    no_pont = list()
    for t in tokens:
        if _HAS_ALNUM.search(t):
            no_pont.append(t)
    return no_pont

tokens_sem_pontuacao = remove_punctuation(tokens_lower)
print(f"Tokens antes: {len(tokens_lower)}\nTokens depois: {len(tokens_sem_pontuacao)}")
print(tokens_sem_pontuacao[:30])

Tokens antes: 235
Tokens depois: 207
['a', 'capivara', 'nome', 'científico', 'hydrochoerus', 'hydrochaeris', 'é', 'uma', 'espécie', 'de', 'mamífero', 'roedor', 'da', 'família', 'caviidae', 'e', 'subfamília', 'hydrochoerinae', 'alguns', 'autores', 'consideram', 'que', 'deva', 'ser', 'classificada', 'em', 'uma', 'família', 'própria', 'está']


### 2.3 Carrega e remove as stopwords
1. O arquivo usado foi `stopwords-pt.txt`

In [ ]:
def load_stopwords(path):
    stopwords = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            word = line.strip().lower()
            if word:
                stopwords.add(word)
    return stopwords

def remove_stopwords(tokens, stopwords):
    no_stopwords = list()
    for t in tokens:
        if t.lower() not in stopwords:
            no_stopwords.append(t)
    
    return no_stopwords

stopwords_pt = load_stopwords(os.path.join("utils/", "stopwords-pt.txt"))
print(f"{len(stopwords_pt)} stopwords carregadas. Exemplos: {sorted(list(stopwords_pt))[:10]}")

tokens_sem_stopwords = remove_stopwords(tokens_sem_pontuacao, stopwords_pt)
print(f"Tokens antes: {len(tokens_sem_pontuacao)}\nTokens depois: {len(tokens_sem_stopwords)}")
print(tokens_sem_stopwords[:30])

212 stopwords carregadas. Exemplos: ['a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aqui', 'aquilo', 'as']
Tokens antes: 207
Tokens depois: 128
['capivara', 'nome', 'científico', 'hydrochoerus', 'hydrochaeris', 'espécie', 'mamífero', 'roedor', 'família', 'caviidae', 'subfamília', 'hydrochoerinae', 'alguns', 'autores', 'consideram', 'deva', 'ser', 'classificada', 'família', 'própria', 'incluída', 'grupo', 'roedores', 'classificam', 'pacas', 'cutias', 'preás', 'porquinho-da-índia', 'ocorre', 'toda']


### 2.4 Tipos -> com e sem pontuaçao

In [127]:
def count_types(tokens):
    return len(set(tokens))

tipos_com_pontuacao = count_types(tokens_lower)
tipos_sem_pontuacao = count_types(tokens_sem_pontuacao)
print(f"Tipos com pontuação: {tipos_com_pontuacao}")
print(f"Tipos sem pontuação: {tipos_sem_pontuacao}")

Tipos com pontuação: 141
Tipos sem pontuação: 136


### Tokens -> com e sem pontuação

In [128]:
def count_tokens(tokens):
    return len(tokens) #desnecessário mas ok

tokens_com_pontuacao_n = count_tokens(tokens_lower)
tokens_sem_pontuacao_n = count_tokens(tokens_sem_pontuacao)
print(f"Tokens com pontuação: {tokens_com_pontuacao_n}")
print(f"Tokens sem pontuação: {tokens_sem_pontuacao_n}")

Tokens com pontuação: 235
Tokens sem pontuação: 207


### 2.6 `n` tipos mais frequentes

In [129]:
def most_frequent_types(tokens, n):
    freq = Counter(tokens)
    return dict(freq.most_common(n))

print("5 tipos mais frequentes (sem pontuação):")
print(most_frequent_types(tokens_sem_pontuacao, 5))

5 tipos mais frequentes (sem pontuação):
{'de': 11, 'e': 8, 'a': 7, '1': 6, 'do': 5}


### 2.7 Frequencia maior que `n`
- Usei n=3 

In [ ]:
def types_above_frequency(tokens, n):
    freq = Counter(tokens)
    return {w: c for w, c in freq.items() if c > n}

n = 3
acima_de_n = types_above_frequency(tokens_sem_pontuacao, n)
print(f"{len(acima_de_n)} tipos com frequência > {n}:")
print(acima_de_n)

10 tipos com frequência > 3:
{'a': 7, 'é': 4, 'uma': 4, 'de': 11, 'e': 8, 'em': 4, 'no': 4, 'do': 5, 'até': 5, '1': 6}


### 2.8 `n` tipos mais raros
- Usei n=5 como exemplo

In [131]:
def rarest_types(tokens, n):
    freq = Counter(tokens)
    ordenado = sorted(freq.items(), key=lambda kv: (kv[1], kv[0]))
    return dict(ordenado[:n])

n = 5
print(f"{n} tipos mais raros:")
print(rarest_types(tokens_sem_pontuacao, n))

5 tipos mais raros:
{'3': 1, '6': 1, '9': 1, 'adaptações': 1, 'adaptável': 1}


### 2.9 - Listar os tipos com frequência menor que `n`

In [132]:
def types_below_frequency(tokens, n):
    freq = Counter(tokens)
    result = {}
    for w, c in freq.items():
        if c < n:
            result[w] = c
            
    return result

n = 2
abaixo_de_n = types_below_frequency(tokens_sem_pontuacao, n)
print(f"{len(abaixo_de_n)} tipos com frequência < {n} (hapax legomena). Exemplos:")
print(dict(list(abaixo_de_n.items())[:10]))

106 tipos com frequência < 2 (hapax legomena). Exemplos:
{'capivara': 1, 'nome': 1, 'científico': 1, 'hydrochoerus': 1, 'hydrochaeris': 1, 'espécie': 1, 'mamífero': 1, 'caviidae': 1, 'subfamília': 1, 'hydrochoerinae': 1}


### Resumo dos resultados

In [133]:
def resumo_pipeline(texto, stopwords, label, n=10):
    tokens = tokenize(texto)
    tokens_l = to_lowercase(tokens)
    tokens_np = remove_punctuation(tokens_l)
    tokens_ns = remove_stopwords(tokens_np, stopwords)

    print(f"**{label} **")
    print(f"tokens (com pontuação): {count_tokens(tokens_l):,}")
    print(f"tipos  (com pontuação): {count_types(tokens_l):,}")
    print(f"tokens (sem pontuação): {count_tokens(tokens_np):,}")
    print(f"tipos  (sem pontuação): {count_types(tokens_np):,}")
    print(f"tokens (sem stopwords): {count_tokens(tokens_ns):,}")
    print(f"top {n} mais frequentes (sem stopwords):")
    print(most_frequent_types(tokens_ns, n))
    print()


# Corpus em português: usamos a lista de stopwords em português
resumo_pipeline(capivara_text, stopwords_pt, "Corpus PT (capivara)")

**Corpus PT (capivara) **
tokens (com pontuação): 235
tipos  (com pontuação): 141
tokens (sem pontuação): 207
tipos  (sem pontuação): 136
tokens (sem stopwords): 128
top 10 mais frequentes (sem stopwords):
{'1': 6, 'ser': 3, '0': 3, 'roedor': 2, 'família': 2, 'm': 2, 'pode': 2, 'pesando': 2, 'kg': 2, '2': 2}



## 4 Corretor ortografico

- O texto de entrada é dividido em tokens pela função `tokenize`
- Cada token que representa uma **palavra** é comparado
  ao vocabulário `V`.
- tokens de pontuacao sao ignorados.
- Se a palavra estiver fora do vocabulario definido, então é calculado a *distância de
  Levenshtein normalizada*. A sugestão é a palavra de `V` com menor
  distância normalizada
- Foi usado a biblioteca `nltk.metrics.distance.edit_distance`.

In [ ]:
VOCABULARY = [
    "O", "A", "Os", "As",
    "homem", "homens", "mulher", "mulheres",
    "é", "são",
    "inteligente", "inteligentes",
    "brilhante", "brilhantes",
]

#_VOCAB_LOWER = {w.lower(): w for w in VOCABULARY}
_VOCAB_LOWER = {}
for w in VOCABULARY:
    _VOCAB_LOWER[w.lower()] = w

_IS_WORD = re.compile(rf"^{_REGEX}$")

def normalized_levenshtein(a, b):
    distance = edit_distance(a, b)
    return distance / max(len(a), len(b))

def suggest_correction(word, vocabulary = VOCABULARY):
    word_lower = word.lower()
    if word_lower in _VOCAB_LOWER:
        return _VOCAB_LOWER[word_lower], 0.0

    best_word, best_distance = None, float("inf")
    for candidate in vocabulary:
        d = normalized_levenshtein(word_lower, candidate.lower())
        if d < best_distance:
            best_distance, best_word = d, candidate
    return best_word, best_distance

#funcao para analisar token por token e sugere correcao baseads no vocabulario (VOCABULARY)
def spell_check(text, vocabulary = VOCABULARY):
    tokens = tokenize(text) #tokeniza com a funcao la de cima
    results = []
    for tok in tokens:
        if _IS_WORD.match(tok): # -> esse _IS_WORD é para checar se é uma palavra valida ou apenas um simbolo/espaço
            suggestion, distance = suggest_correction(tok, vocabulary)
            results.append((tok, suggestion, round(distance, 3)))
        else:
            results.append((tok, tok, 0.0)) # se nao é palavra, assume que esta correto e define distancia como 0.0
    return results

ex_frases = [
    "o homem é inteligentr",
    "Os homens sõ brilhantess",
    "As muleres são brilantes",
    "A milher é inteliggente",
]

for frase in ex_frases:
    print(f"Frase original: {frase!r}")
    corrigida = []
    for token, sugestao, dist in spell_check(frase):
        corrigida.append(sugestao)
    print("Frase corrigida:", " ".join(corrigida))
    print()


Frase original: 'o homem é inteligentr'
Frase corrigida: O homem é inteligente

Frase original: 'Os homens sõ brilhantess'
Frase corrigida: Os homens são brilhantes

Frase original: 'As muleres são brilantes'
Frase corrigida: As mulheres são brilhantes

Frase original: 'A milher é inteliggente'
Frase corrigida: A mulher é inteligente

